# 🛍️ Customer Review Sentiment Analyzer

**Project:** Customer Review Sentiment Analyzer  
**Repository:** ML-CaPsule | GSSoC'26  
**Issue:** #1921  
**Category:** NLP · Sentiment Analysis · Beginner-Friendly  

---

## 1. 📖 Project Introduction

### What is Sentiment Analysis?

**Sentiment Analysis** (also called **Opinion Mining**) is a Natural Language Processing (NLP) technique used to determine whether a piece of text expresses a **positive**, **negative**, or **neutral** emotion or opinion.

It is one of the most widely used applications of NLP and machine learning in the real world.

### Why is Customer Review Classification Useful?

In the era of e-commerce, millions of customers leave reviews every day. Manually reading and categorizing them is impossible at scale. Automated sentiment analysis helps businesses:

- ⚡ **Quickly identify** unhappy customers and address their concerns
- 📈 **Track product quality** trends over time
- 🎯 **Improve recommendations** based on sentiment signals
- 🏷️ **Automate tagging** of reviews as positive, neutral, or negative
- 📊 **Generate insights** for product teams and marketers

### Project Goal

In this notebook, we will:
1. Load and explore an Amazon product review dataset
2. Preprocess raw review text using NLP techniques
3. Create sentiment labels from star ratings
4. Extract features using **TF-IDF Vectorization**
5. Train a **Logistic Regression** classifier
6. Evaluate the model and make interactive predictions

---

## 2. 📦 Import Libraries

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import re
import string
import warnings

# ── Data Manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ── Settings ──────────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')        # Suppress harmless warnings
np.random.seed(42)                       # Reproducibility

# Download required NLTK data (runs silently if already downloaded)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print('✅ All libraries imported successfully!')

## 3. 📂 Load Dataset

We use a sample **Amazon product review** dataset. The dataset contains two key columns:
- **`reviewText`** — the full text of the customer review
- **`overall`** — the star rating given by the customer (1 to 5)

> **Dataset Source:** This notebook uses a curated sample dataset that is built in-line for portability. To use the full Amazon Reviews dataset, you can download it from [Kaggle - Amazon Product Reviews](https://www.kaggle.com/datasets/bittlingmayer/amazonreviews) or the [UCSD Amazon Review Data repository](https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/).

A balanced set of **~500 sample reviews** (≈167 per class) is created programmatically so this notebook runs out of the box.

In [ ]:
def create_sample_dataset():
    """
    Creates a balanced sample Amazon product review dataset.
    Returns a DataFrame with 'reviewText' and 'overall' columns.
    """
    # ── Positive Reviews (4–5 stars) ──────────────────────────────────────────
    positive_reviews = [
        "This product exceeded all my expectations. Absolutely love it!",
        "Amazing quality, fast shipping. Highly recommend to everyone!",
        "Best purchase I have made this year. Worth every penny.",
        "Fantastic product, works exactly as described. Very happy!",
        "Excellent build quality and great value for money.",
        "I am extremely satisfied with this product. It is perfect.",
        "Outstanding performance and beautiful design. Love it!",
        "Super happy with this purchase. Works perfectly right out of the box.",
        "Great product, great price. Arrived on time. Five stars!",
        "This is exactly what I was looking for. Excellent quality!",
        "Incredible product, surpassed all expectations. Would buy again!",
        "Very well made, durable and elegant. Perfect gift idea.",
        "Top notch quality, customer service was wonderful too. Highly recommend.",
        "This product is simply amazing. It changed my daily routine for the better.",
        "Easy to use, high quality materials, and looks great. No complaints!",
        "Phenomenal product at an affordable price. Very satisfied with my purchase.",
        "Arrived quickly and works beautifully. Would definitely purchase again.",
        "I have been using this for a month and it is holding up great!",
        "The quality is far better than I anticipated at this price point.",
        "Wonderful product! Easy to set up and works like a charm.",
        "This is a gem of a product. It does exactly what it promises.",
        "Super impressed with the build quality and attention to detail.",
        "Bought as a gift and the recipient absolutely loved it!",
        "Everything about this product is top tier. Packaging, quality, performance.",
        "Five stars without hesitation. I am completely blown away.",
        "Works flawlessly. So glad I chose this over the alternatives.",
        "Exceeded expectations in every possible way. True value for money.",
        "Hands down the best product in this category. A must buy!",
        "Gorgeous design, solid build. My whole family loves it!",
        "Delivery was fast and the product quality is beyond impressive.",
        "Absolutely perfect for my needs. Could not be happier with this purchase.",
        "I read hundreds of reviews before buying and they were all right. This is great!",
        "The product photos do not do it justice. It looks even better in person.",
        "Customer support was helpful and the product itself is amazing.",
        "This made my life so much easier. Totally worth the investment.",
        "Just as advertised! Great quality and very easy to use.",
        "I love this product so much I bought a second one as a gift.",
        "Well-packaged, arrived on time, and works better than expected.",
        "The performance of this product is remarkable for the price.",
        "I was skeptical at first but this has become one of my favorite purchases.",
        "No issues whatsoever. Everything works as it should. Very pleased.",
        "An absolute bargain for the quality you get. Strongly recommend!",
        "This product is genuinely impressive. I keep recommending it to friends.",
        "Sturdy, reliable, and user-friendly. Exactly what I needed.",
        "Clean, elegant, and functional. The perfect combination!",
        "Would rate it 6 stars if I could. An exceptional product!",
        "So easy to use even my grandparents figured it out immediately.",
        "Great packaging, arrived in perfect condition, fantastic quality.",
        "Product quality is exceptional and it works perfectly. Very happy!",
        "This is exactly what every customer review said. Just fantastic!",
        "I have tried many similar products and this is hands down the best.",
        "Setup was a breeze and the results are impressive. Totally satisfied!",
        "The product works as advertised. Solid build. Highly satisfied.",
        "Such a high quality product at a very reasonable price point.",
        "Very happy with my purchase. Will definitely buy from this brand again.",
        "Works better than the more expensive alternatives I have tried.",
        "Delivery on time, packaging excellent, and the product itself is superb!",
        "Clean design, great functionality. My whole team loves using it.",
        "Reliable and efficient. Has not let me down even once.",
        "One of the best investments I have made. Highly recommended!",
        "Stellar product in every dimension. I am completely satisfied!",
        "Wonderful! Works exactly as described and looks great too!",
        "Exceptional quality and rapid delivery. Exceeded my expectations entirely.",
        "I cannot say enough good things about this product. It is simply superb.",
        "Perfect size, perfect quality, and arrived perfectly packaged.",
        "This product really delivers on its promise. Truly impressed.",
        "Love everything about this product. Already recommended to 5 friends!",
        "Works great, looks great. Could not ask for more at this price.",
        "Very durable and well-crafted. Will last for years.",
        "Outstanding customer experience from order to delivery. Product is top notch.",
        "Simple to use and very effective. Saved me a lot of time.",
        "Beautifully designed and extremely functional. Rare combination!",
        "I was amazed at how well this product works. Top quality!",
        "Works exactly as described, maybe even better. Very impressed!",
        "Great for everyday use. Lightweight yet durable. Very happy!",
        "Could not be happier with this product. Everything about it is great!",
        "Very satisfied with the purchase. The quality is outstanding for the price.",
        "This product is everything I needed and more. Absolutely love it!",
        "Premium feel and performance. You get a lot of value for the price.",
        "Easy to set up, intuitive to use, and works perfectly. Great buy!",
        "The construction quality is superb. Feels like it will last forever.",
        "Unbelievably good quality for the price. I am very impressed.",
        "Does what it promises and does it well. No complaints at all!",
        "Brilliant product with no flaws. Highly recommend to anyone looking.",
        "This is a quality product through and through. Completely satisfied!",
        "Happy beyond words. This product has made such a difference in my life.",
        "Simply excellent. The product quality speaks for itself.",
        "Well worth the money. Performs above expectations on all counts.",
        "Incredible value. I would buy this again without hesitation.",
        "So glad I found this product. Exactly what I was looking for!",
        "My expectations were already high and this product still exceeded them.",
        "Works beautifully and looks great. The perfect purchase decision.",
        "Amazing product with zero flaws. Shipping was fast too. Five stars!",
        "Best quality I have seen at this price range. Highly impressed!",
        "I am thrilled with this product. It does everything it claims and more!",
        "Solid product, great customer service, and delivered ahead of schedule!",
        "This product makes my day easier every single day. So thankful I bought it.",
        "Top quality and very reliable. I am truly impressed by this purchase.",
        "Excellent craftsmanship and attention to detail. Very happy customer!",
        "Just what I was looking for! Excellent quality and super fast delivery.",
        "Absolutely brilliant! Does exactly what it says. Would buy again!",
        "Very happy with this product. It is exactly as described and works great.",
        "The quality and performance of this product surprised me in the best way.",
        "Product is exactly as shown and works better than expected. Five stars!",
        "Phenomenal quality at a fair price. Cannot ask for anything more.",
        "Runs flawlessly. I love it! Would recommend to family and friends.",
        "Great product backed by excellent customer service. Very happy overall.",
        "Impressively well-made and the performance matches the price perfectly.",
        "Such an amazing product. It does exactly what it says it will do.",
        "Well constructed, easy to use, and genuinely helpful. Love it!",
        "This product is a real winner. Excellent in every single way.",
        "Happy with every aspect of this purchase. Quality, delivery, and support.",
        "Smooth, efficient, and well-designed. I am beyond satisfied!",
        "Absolutely love this product. Works like a dream every single time.",
        "This was a perfect purchase decision. Would recommend it to everyone!",
        "High quality and very durable. Cannot ask for a better product.",
        "Exceeded all my expectations. I could not be happier with this product!",
        "Works exactly as advertised. The perfect addition to my collection!",
        "Very high quality product that performs exactly as described. Highly satisfied.",
        "I am completely blown away by the quality of this product. Five stars!",
        "Product quality is first class. Works perfectly and looks stunning.",
        "Simply the best product in its category. Highly recommend buying it!",
        "Very happy with this. Excellent quality and great performance overall.",
        "So pleased with this purchase. Exactly what I needed and then some!",
        "No complaints whatsoever. This product is truly excellent in every way.",
        "Works perfectly and arrived quickly. Excellent all around product!",
        "This product is a delight to use every day. Highly recommended!",
        "Premium quality and very effective. Well worth every penny I spent.",
        "Impressed by the quality and very satisfied with my purchase overall.",
        "This product lives up to the hype. Simply fantastic. Five stars!",
        "Perfect product for my needs. Great value and excellent quality.",
        "This is a top quality product and I am very pleased with my purchase.",
        "Received in perfect condition and works flawlessly. Very happy!",
        "I love how well this product works. It is absolutely fantastic!",
        "Quality is unmatched for the price. An excellent buy through and through.",
        "Works like a champ. Fast delivery and excellent packaging. Five stars!",
        "This product is superb. I am very impressed by its quality and performance.",
        "Loved it from the moment I opened the package. Simply the best!",
        "Perfect in every way. Would give it more than five stars if I could!",
        "An exceptional product that delivers on every single promise made.",
        "Extremely satisfied. This product is everything the description says it is.",
        "High quality, easy to use, and very effective. Five stars without doubt!",
        "I cannot recommend this product highly enough. Truly outstanding.",
        "Best product I have purchased in a long time. Absolutely love it!",
        "Works great and looks amazing. This is genuinely a perfect product.",
        "Stellar quality, fast delivery, and great value. Highly recommended!",
        "Very durable and effective. I am very happy I chose this product.",
        "Great experience from start to finish. Love this product so much!",
        "Totally impressed with the quality. A fantastic product overall!",
        "This product has made my life so much more convenient. Thank you!",
        "Quality and performance are excellent. I could not be more satisfied!",
        "Extremely pleased with this product. Worth every single penny I paid.",
        "This product is a winner! Excellent quality and superb performance.",
        "Brilliantly designed and a joy to use. I am thrilled with this purchase!",
        "Durable, stylish, and highly effective. An absolutely excellent buy."
    ]

    # ── Neutral Reviews (3 stars) ─────────────────────────────────────────────
    neutral_reviews = [
        "It is okay, not great, not terrible. Does the job.",
        "Average quality. Nothing special but works as expected.",
        "Decent product for the price. Nothing really stands out.",
        "It works fine, but I expected better based on the description.",
        "Middle of the road. Gets the job done but won't wow you.",
        "Acceptable quality, but I have seen better. Average experience.",
        "The product is fine. Not amazing, but not disappointing either.",
        "Somewhat useful. Not the best, but not the worst either.",
        "Average product with average performance. No strong feelings.",
        "Does what it should, nothing more. A mediocre experience overall.",
        "Moderate quality for the price point. Neither impressed nor upset.",
        "It is acceptable. Not exactly what I wanted but it works okay.",
        "I have mixed feelings about this product. Some good, some bad.",
        "Works as described but is not particularly impressive in any way.",
        "Reasonable quality but I expected a bit more for the money.",
        "Neither here nor there. A functional product with no wow factor.",
        "It gets three stars from me. Not bad but not great either.",
        "Just okay. I would neither strongly recommend nor discourage it.",
        "This product is average. Good enough to use but nothing special.",
        "Reasonable purchase, but there is room for improvement.",
        "Not bad. It does what it claims but it is not exciting.",
        "Mediocre product. Serviceable but unremarkable in every way.",
        "Works, but just barely meets my expectations. Average experience.",
        "Fair product at a fair price. You get exactly what you pay for.",
        "The product functions correctly but lacks the quality I hoped for.",
        "So-so. This product is definitely not the best but it is usable.",
        "Three stars feels right. Not disappointed but not thrilled either.",
        "Works adequately. There are better options but also worse ones.",
        "A typical product in this price range. Nothing to get excited about.",
        "Mixed bag. Has some nice features but let down in other areas.",
        "This is an average product. Has some good aspects and some bad.",
        "Not sure how I feel about this. Some days it seems great, others less so.",
        "Works as advertised. Nothing more, nothing less. Average all around.",
        "Satisfactory product. Does what it needs to do without standing out.",
        "It functions well enough but there is definitely room for improvement.",
        "A middle-of-the-pack product. It works but does not impress.",
        "Reasonable quality but I have had better experiences with similar products.",
        "Somewhat satisfied. The product works but does not exceed expectations.",
        "The quality is fair. You get what you pay for with this product.",
        "Average quality product that fulfills its basic purpose. That is all.",
        "Not impressed, but not disappointed either. Pretty much what I expected.",
        "This product is fine, I guess. It does the job but nothing noteworthy.",
        "Okay product. Not the worst I have seen but definitely not the best.",
        "Slightly above average. Some features are good, others are lacking.",
        "I feel indifferent about this product. It is neither great nor terrible.",
        "Expected better but it will do. An unremarkable, average product overall.",
        "It gets the job done but barely. Not sure if I would purchase again.",
        "It is usable but feels generic and lacks any standout qualities.",
        "Three out of five is fair. A middle-ground product without any real highlights.",
        "Works fine on most days. Has some limitations but nothing deal-breaking.",
        "The product is functional. Not great, just functional. Fair enough.",
        "Does not disappoint but does not impress either. A standard product.",
        "This product meets the bare minimum requirements. Just adequate.",
        "Not exactly what I was hoping for but it will serve its purpose.",
        "Perfectly average product. Nothing remarkable about it at all.",
        "Somewhat useful but the novelty wore off quickly. Average rating.",
        "It is okay. I would not buy again but I would not return it either.",
        "Performance is acceptable. Could be better but it is also not terrible.",
        "Good enough to get by with. But there are probably better options.",
        "A standard purchase with no real highs or lows. Average all round.",
        "Neither love it nor hate it. It works and that is enough I suppose.",
        "Works about 80% of the time. Not perfect but not a disaster.",
        "The packaging was nice but the product itself is pretty average.",
        "I have used better products before. This one is mediocre by comparison.",
        "Decent enough product. I am not unhappy but not particularly pleased.",
        "Mixed feelings here. The product has potential but does not deliver fully.",
        "An uninspiring product that gets the job done without any flair.",
        "The product is fine. Nothing wrong with it, but nothing special either.",
        "Three stars because it works but does not deliver on its promise.",
        "Not outstanding but not bad either. Right in the middle of the road.",
        "It serves its purpose. Would not rave about it but would not complain much.",
        "Average product at an average price. Nothing particularly noteworthy.",
        "This product is just okay. Works as expected but does not stand out.",
        "It could be better but at this price I suppose it is acceptable.",
        "Not bad but not good. Kind of in the middle. Unremarkable overall.",
        "I am on the fence about this product. Has pros and cons.",
        "Meets basic requirements. Does not impress but does not frustrate.",
        "Average performance and average build quality. Nothing to write home about.",
        "The product does its job but lacks the polish I expected.",
        "I do not love it and I do not hate it. A very average experience.",
        "Does what it claims but just barely. Three stars seems right.",
        "Not a bad product but not good enough to buy again. Very average.",
        "It works most of the time. Some minor issues but generally functional.",
        "Three stars for a product that is perfectly average in every way.",
        "I am kind of neutral on this. It works but it is not exciting.",
        "Solid product but nothing exciting. Does what it needs to do.",
        "Works as described. Not exceptional but not problematic either.",
        "Acceptable purchase for the price but lacks that wow factor.",
        "Okay product. Not the best option but is not the worst either.",
        "Pretty average in all aspects. Gets the job done at least.",
        "Not particularly impressed but also not bothered. Average product.",
        "I expected more but at least the product is functional. Just okay.",
        "The product delivers an average performance. Nothing really stands out.",
        "It does the job. I feel somewhat neutral about this overall purchase.",
        "Very average, very unremarkable. A three-star product through and through.",
        "Some features work well, some do not. Overall a mixed but average experience.",
        "Average in every way. Just an okay product that does its basic job.",
        "It is usable. I would call it just acceptable for the money spent.",
        "Works alright for the most part. Not something I would enthusiastically recommend.",
        "A fine product if you are not looking for anything special.",
        "This product does what it is supposed to. Nothing less, nothing more.",
        "Three stars. Not thrilled but not disappointed. Just neutral overall.",
        "The quality is about what you would expect at this price. Just average.",
        "Okay experience overall. Works adequately but has room to grow.",
        "An average product that performs averagely. Nothing to get excited about.",
        "The product is functional but not exceptional in any meaningful way.",
        "I am not unhappy, but not particularly satisfied either. Average rating.",
        "It performs adequately. Not groundbreaking but also not useless.",
        "Three stars seems fair. Some good points, some bad, averages out.",
        "I think it is an okay product. Does the job without any drama.",
        "This is a take-it-or-leave-it product. Works, but nothing special.",
        "Pretty standard product, nothing to get excited about. Average all around.",
        "Fair quality. You get a product that does its job without excelling.",
        "I see both positives and negatives in equal measure. Average overall.",
        "Works adequately for everyday use. Not remarkable but not a failure.",
        "This product is fine for what it is. Not great, not bad. Just okay.",
        "Average quality, average performance, average everything. Three stars.",
        "Does exactly what it says. I just expected a slightly higher quality.",
        "Neutral feelings about this product. It functions but lacks distinction.",
        "Satisfactory performance, nothing special. Exactly what average looks like.",
        "It works, so I cannot complain much. But it does not exceed expectations.",
        "The product is average. I give it three stars because it is not bad.",
        "Somewhat satisfied. The product works but there is definitely room for more.",
        "Nothing to rave about. A perfectly average product at an average price.",
        "The product is a bit underwhelming but functional enough for basic use.",
        "A mediocre product that meets my needs but does not stand out at all.",
        "Three stars. This product delivers acceptable performance nothing more.",
        "Very average. Works as expected but nothing exceptional about it.",
        "I find this product to be simply average. Not impressive, not terrible.",
        "Adequate product. Not the worst purchase but definitely not the best.",
        "It is fine. An ordinary product in an ordinary category. Average.",
        "Functional and basic. Meets minimum requirements without any surprise.",
        "This product earns a solid three stars. Not good, not bad, just average.",
        "A product that just barely meets expectations. Functional, but that is it.",
        "Works on most occasions. Has limitations but is overall acceptable.",
        "The performance is fair but not impressive. Average product overall.",
        "Three stars. I am not going to recommend it but also will not warn against it.",
        "A middle-ground product that does what it should, nothing more.",
        "I rate this an average three stars. Functional but ultimately forgettable.",
        "The quality is acceptable. Not mind-blowing but also not bad at all.",
        "Average experience. Neither a disappointment nor a happy surprise.",
        "It works fine. A standard product with standard results. Three stars.",
        "This is an ordinary product. It does its job but lacks any excitement.",
        "I feel neither strongly positive nor negative about this product.",
        "It is a decent product. I have no strong feelings one way or the other.",
        "The product gets the job done but without any outstanding qualities.",
        "It's an average product. Not the best and not the worst option available.",
        "This product is okay. I am not unhappy but also not particularly pleased."
    ]

    # ── Negative Reviews (1–2 stars) ──────────────────────────────────────────
    negative_reviews = [
        "Terrible product. Broke after two days. Complete waste of money!",
        "Absolute garbage. Do not waste your money on this junk product.",
        "Worst purchase I have ever made. Nothing works as advertised.",
        "Complete disappointment. The quality is shockingly poor.",
        "Do not buy this! It is a scam. Broke within a week of purchase.",
        "Horrible experience. The product is cheaply made and dysfunctional.",
        "Extremely disappointed. This product is nothing like the description.",
        "Returned immediately. The product was defective right out of the box.",
        "Save your money and avoid this at all costs. Absolutely terrible.",
        "Very poor quality. Fell apart in just days. Not worth a single penny.",
        "Waste of money. The product does not do what it claims to do at all.",
        "Would give zero stars if I could. Appalling quality and service.",
        "Cheaply made and breaks easily. Regret buying this completely.",
        "A disgrace of a product. Do not be fooled by the photos. Terrible.",
        "Does not work at all. Complete and utter waste of time and money.",
        "I am furious. This product broke on the very first day of use.",
        "Horrible quality. The product looks nothing like the listing photos.",
        "Utter disappointment from start to finish. I want my money back.",
        "The worst product I have ever bought. Completely useless garbage.",
        "Cheap materials, poor build quality, terrible performance. Avoid!",
        "This product is a fraud. The quality is atrocious and it broke quickly.",
        "Do not buy this product. It is shoddily made and does not work.",
        "I am absolutely disgusted. This product is dangerous and defective.",
        "The product is a joke. It literally fell apart as I was unboxing it.",
        "So disappointed. Spent good money on this and it is completely useless.",
        "Pathetic quality. Stopped working after just a few hours of use.",
        "Completely malfunctions every time. What a waste of my hard earned money.",
        "This is junk. Poor quality, poor design, poor performance. Avoid it!",
        "Huge disappointment. Does not even come close to what was promised.",
        "The worst purchase I have made online. This product is absolutely terrible.",
        "Shoddy craftsmanship and unreliable performance. I deeply regret this buy.",
        "Defective product. The seller sent me a broken item. Very unhappy!",
        "The quality is absolutely unacceptable. This is pure garbage.",
        "I cannot believe they are selling this. It is completely defective!",
        "Misleading description. The product is nothing like what was advertised.",
        "This product has caused me more problems than it solved. Very poor!",
        "Do yourself a favor and avoid this product entirely. Terrible quality!",
        "Absolute disaster. The product broke and customer service was useless.",
        "One star is too generous. This product is a complete and utter failure.",
        "The lowest quality product I have ever had the misfortune of purchasing.",
        "Do not be fooled by the reviews. This product is terrible and unreliable.",
        "Wasted my money on this junk. The product stopped working after one use.",
        "This is a defective and misleading product. I want a full refund.",
        "Dreadful quality. This product is flimsy, unreliable, and useless.",
        "Appalling product. The quality is nothing short of disgraceful.",
        "I was completely deceived by the product listing. It is awful.",
        "Do not buy. This product is cheap, poorly made, and just terrible.",
        "A complete rip-off. The product does not work and is very fragile.",
        "Awful in every possible way. I deeply regret purchasing this product.",
        "Extremely poor quality. Fell apart on first use. Deeply disappointed.",
        "This product is garbage. Breaks immediately and does not work at all.",
        "Avoid this product like the plague. It is simply terrible.",
        "Very angry with this purchase. The product is defective and useless.",
        "The product is completely nonfunctional. Save your money and look elsewhere.",
        "This product is a complete waste. It does not work and is poorly made.",
        "Terrible experience. The product broke immediately and was clearly defective.",
        "I have never been so disappointed with a purchase. This is truly awful.",
        "The quality is shockingly low for the price charged. Do not buy this!",
        "The product does not function at all. A total and complete waste of money.",
        "Garbage! It broke on the second day. Absolutely unacceptable quality.",
        "Worst product I have ever used. The quality is embarrassingly poor.",
        "I paid good money for this junk. It stopped working after three uses.",
        "Defective from the start. Returned it immediately. Terrible product.",
        "Extremely poor build quality. Would not even gift this to an enemy.",
        "This product has zero redeeming qualities. A total disappointment.",
        "Overpriced rubbish. Does not work as claimed. Absolutely terrible.",
        "Cheap, fragile, and completely useless. I want my money back!",
        "The product arrived broken and the return process was a nightmare.",
        "Very poor quality control. My product was defective right from day one.",
        "This is the worst thing I have ever bought online. Total garbage!",
        "Terrible product with terrible customer service. Avoid at all costs!",
        "This is junk. Breaks immediately and is completely useless.",
        "Absolute rubbish. Does not work as advertised. Total waste of money.",
        "I am so frustrated with this product. It has been a nightmare to use.",
        "Poor quality, does not work, and the seller ignored my complaint. Terrible!",
        "This product failed within hours of use. Absolutely worthless.",
        "I cannot express how disappointed I am. This product is dreadful.",
        "Horrific quality and performance. I would never recommend this to anyone.",
        "The product description is completely misleading. It is a terrible product.",
        "Fell apart within days of use. Zero quality control. Do not buy!",
        "I have used many products in this category. This is by far the worst.",
        "Complete garbage. Does not work as described. Total waste of money.",
        "Do not waste your hard earned money on this terrible product. Avoid!",
        "The product is an embarrassment. Terrible quality and poor performance.",
        "I want a refund. This product is defective and completely useless to me.",
        "Appalling in every way. Quality is shocking and it does not work at all.",
        "This is the biggest waste of money I have spent. Truly terrible product.",
        "Absolutely dreadful. Stopped working within 24 hours. Terrible quality!",
        "Do not buy this. It is an inferior product with no redeeming features.",
        "The product is substandard and the photos were completely misleading.",
        "Worst possible purchase experience. Product is defective and useless!",
        "The quality of this product is an absolute disgrace. Do not buy it!",
        "I am outraged by the poor quality of this product. Total money waste.",
        "This product is dangerous! It stopped working and produced smoke.",
        "Nothing works as described. A fraudulent listing for a terrible product.",
        "This is absolutely terrible. Broke on first use and is useless.",
        "The product is completely broken. I was sent a non-functional product.",
        "Junk product. Stop selling this garbage to unsuspecting customers!",
        "Broke immediately and customer service was not helpful at all. Terrible!",
        "Fraud! The product arrived broken and the seller refused to refund me.",
        "Cheaply made and completely unreliable. Deeply disappointed in this.",
        "This product is pure garbage. Do not waste your money on it!",
        "The worst product I have ever owned. A complete disappointment!",
        "Terrible, terrible product. Zero quality and zero performance. Avoid!",
        "This is garbage. It broke after one use. Complete waste of money.",
        "Cheap and flimsy. Does not perform as advertised. Terrible purchase.",
        "Absolutely horrendous product. Failed within hours. Very disappointed.",
        "I regret this purchase deeply. The product is defective and cheap.",
        "A total disaster. Poor quality and the product does not work at all.",
        "The listing was misleading and the product is shockingly poor quality.",
        "Defective, useless, and poorly made. The worst product I have purchased.",
        "Garbage product. I demand a refund. It broke the very first day.",
        "Total waste of money. The product is broken and does not do anything.",
        "I am completely appalled by the quality of this product. Just terrible.",
        "Product stopped working immediately. Terrible quality control. Avoid!",
        "This product is a complete failure. Cheap, nasty, and non-functional.",
        "Very unhappy with this product. It broke and now I am stuck with it.",
        "The product is everything I feared. Cheap, flimsy, and useless.",
        "What a disappointment. This product does not work and feels very cheap.",
        "I cannot believe this is even being sold. It is absolutely terrible!",
        "This product is a scam. It does not work at all. Complete garbage!",
        "Bought this and regretted it instantly. The quality is atrociously bad.",
        "Extremely disappointed. The product is defective and customer service is useless.",
        "This product is nothing short of a disaster. I want my money back!",
        "One star is too many for this product. It is absolutely worthless.",
        "Do not buy this under any circumstances. It is terrible quality garbage.",
        "The product failed immediately and customer service did not help at all.",
        "Terrible, useless, and cheaply constructed. A completely bad purchase.",
        "This product is a severe disappointment. Quality is absolutely dreadful.",
        "The product does not work at all. Absolute garbage and waste of money.",
        "I am furious. This product is defective and was misleadingly described.",
        "Awful product and awful service. Do not waste your money on this item.",
        "This is shamefully bad quality. I cannot believe anyone sells this junk.",
        "I am appalled by the quality of this product. Never again!",
        "Embarrassingly poor quality. It broke immediately. Do not buy.",
        "Total garbage. Broke on first use. Absolutely useless and worthless!",
        "A pathetically poor product that does not function. Very angry.",
        "The quality of this product is an absolute embarrassment. Terrible!",
        "What a waste! This product stopped working right out of the packaging.",
        "Horrendous product quality. Broken on arrival. Completely unacceptable!",
        "This product is a scam and a disappointment. Save your money!",
        "Cheap, flimsy rubbish. I want a full refund. Absolutely terrible!",
        "This is a dreadfully made product with no positive qualities.",
        "The product is fraudulent. It does not work and the photos are fake.",
        "Cannot believe the audacity of selling something this poor quality.",
        "This product has broken my trust in online shopping. Truly terrible.",
        "I cannot stress enough how terrible this product truly is. Avoid!",
        "Disgracefully poor product. I have never been more let down by a purchase.",
        "Do not make the mistake of buying this. It is absolute garbage.",
        "The product is a disaster waiting to happen. Avoid it entirely.",
        "Truly the worst product I have ever had the misfortune of purchasing."
    ]

    # Assign star ratings based on sentiment category
    positive_ratings = np.random.choice([4, 5], size=len(positive_reviews))
    neutral_ratings  = np.full(len(neutral_reviews), 3)
    negative_ratings = np.random.choice([1, 2], size=len(negative_reviews))

    # Combine into a single DataFrame
    all_reviews = positive_reviews + neutral_reviews + negative_reviews
    all_ratings = list(positive_ratings) + list(neutral_ratings) + list(negative_ratings)

    df = pd.DataFrame({'reviewText': all_reviews, 'overall': all_ratings})

    # Shuffle and reset index
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df


# Load the dataset
df = create_sample_dataset()

print('✅ Dataset loaded successfully!')
print(f'Total records: {len(df)}')
df.head()

## 4. 🔍 Exploratory Data Analysis (EDA)

Before building the model, let us understand the dataset better.

In [ ]:
# ── 4.1 Dataset Shape ─────────────────────────────────────────────────────────
print('=' * 50)
print('📐 DATASET SHAPE')
print('=' * 50)
print(f'Rows    : {df.shape[0]}')
print(f'Columns : {df.shape[1]}')
print(f'Columns : {list(df.columns)}')

In [ ]:
# ── 4.2 Missing Values ────────────────────────────────────────────────────────
print('=' * 50)
print('🔍 MISSING VALUES')
print('=' * 50)
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# ── 4.3 Data Types and Basic Info ─────────────────────────────────────────────
print('=' * 50)
print('📋 DATA TYPES')
print('=' * 50)
df.dtypes

In [ ]:
# ── 4.4 Distribution of Ratings ───────────────────────────────────────────────
print('=' * 50)
print('⭐ RATING DISTRIBUTION')
print('=' * 50)
print(df['overall'].value_counts().sort_index())

# Plot rating distribution
fig, ax = plt.subplots(figsize=(8, 5))

rating_counts = df['overall'].value_counts().sort_index()
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
bars = ax.bar(rating_counts.index, rating_counts.values, color=colors, edgecolor='black', linewidth=0.8)

# Add value labels on bars
for bar, count in zip(bars, rating_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Distribution of Star Ratings', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Star Rating', fontsize=13)
ax.set_ylabel('Number of Reviews', fontsize=13)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1 ⭐', '2 ⭐', '3 ⭐', '4 ⭐', '5 ⭐'], fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#ffffff')

plt.tight_layout()
plt.savefig('Screenshots/rating_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Rating distribution plot saved.')

In [ ]:
# ── 4.5 Sample Reviews ────────────────────────────────────────────────────────
print('=' * 50)
print('📝 SAMPLE REVIEWS')
print('=' * 50)

for rating in [5, 3, 1]:
    sample = df[df['overall'] == rating]['reviewText'].iloc[0]
    print(f'\n⭐ Rating {rating}: "{sample[:120]}..."' if len(sample) > 120 else f'\n⭐ Rating {rating}: "{sample}"')

## 5. 🧹 Data Preprocessing

Raw text contains noise (URLs, HTML tags, punctuation, numbers, stopwords) that reduces model performance. We will clean the text using a multi-step preprocessing pipeline.

### Preprocessing Steps:
1. **Remove URLs** — URLs carry no sentiment information
2. **Remove HTML tags** — Strip any `<tag>` markup
3. **Remove punctuation** — Punctuation does not carry semantic meaning for BoW models
4. **Remove numbers** — Pure numbers rarely indicate sentiment
5. **Convert to lowercase** — Normalize case for consistent matching
6. **Remove extra whitespace** — Clean up spacing
7. **Remove stopwords** — Common words like 'the', 'is', 'and' add noise
8. **Lemmatization** — Reduce words to their base form (e.g., 'running' → 'run')

In [ ]:
# Initialize NLP tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def remove_urls(text):
    """Remove URLs starting with http, https, or www."""
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)


def remove_html_tags(text):
    """Remove HTML tags such as <br>, <p>, <div>, etc."""
    return re.sub(r'<.*?>', '', text)


def remove_punctuation(text):
    """Remove all punctuation characters from the text."""
    return text.translate(str.maketrans('', '', string.punctuation))


def remove_numbers(text):
    """Remove all standalone numeric characters from the text."""
    return re.sub(r'\d+', '', text)


def to_lowercase(text):
    """Convert all characters in text to lowercase."""
    return text.lower()


def remove_extra_spaces(text):
    """Collapse multiple consecutive spaces into a single space."""
    return re.sub(r'\s+', ' ', text).strip()


def remove_stopwords(text):
    """Remove common English stopwords from the tokenized text."""
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered_tokens)


def lemmatize_text(text):
    """Reduce each word to its base lemma form (e.g., 'running' → 'run')."""
    tokens = text.split()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(lemmatized_tokens)


def preprocess_text(text):
    """
    Full text preprocessing pipeline.
    Applies all cleaning steps in the correct order:
      1. Remove URLs
      2. Remove HTML tags
      3. Remove punctuation
      4. Remove numbers
      5. Convert to lowercase
      6. Remove extra spaces
      7. Remove stopwords
      8. Lemmatize
    """
    text = remove_urls(text)
    text = remove_html_tags(text)
    text = remove_punctuation(text)
    text = remove_numbers(text)
    text = to_lowercase(text)
    text = remove_extra_spaces(text)
    text = remove_stopwords(text)
    text = lemmatize_text(text)
    return text


print('✅ Preprocessing functions defined.')

# Demonstrate preprocessing on one example
example = "This product is <b>absolutely</b> amazing! Visit https://example.com for more info. Rated 5/5 stars!!!"
print(f'\n📌 Example Input : {example}')
print(f'✅ After Cleaning: {preprocess_text(example)}')

In [ ]:
# Apply preprocessing to the entire dataset
print('⏳ Preprocessing all reviews... (may take a few seconds)')
df['cleanedText'] = df['reviewText'].apply(preprocess_text)
print('✅ Preprocessing complete!')

# Show before and after comparison
print('\n📋 Before vs. After Preprocessing:')
print('─' * 80)
for i in range(3):
    print(f'ORIGINAL : {df["reviewText"].iloc[i]}')
    print(f'CLEANED  : {df["cleanedText"].iloc[i]}')
    print('─' * 80)

## 6. 🏷️ Label Creation

We map the numeric star ratings to three sentiment categories:

| Rating    | Sentiment |
|-----------|-----------|
| 1–2 ⭐   | Negative  |
| 3 ⭐      | Neutral   |
| 4–5 ⭐   | Positive  |

In [ ]:
def rating_to_sentiment(rating):
    """
    Convert a numeric star rating into a sentiment label.
    - 1–2 stars  → 'Negative'
    - 3 stars    → 'Neutral'
    - 4–5 stars  → 'Positive'
    """
    if rating <= 2:
        return 'Negative'
    elif rating == 3:
        return 'Neutral'
    else:
        return 'Positive'


# Apply label creation
df['sentiment'] = df['overall'].apply(rating_to_sentiment)

print('✅ Sentiment labels created!')
print('\n📊 Label Distribution:')
print(df['sentiment'].value_counts())

# Visualize sentiment distribution
sentiment_counts = df['sentiment'].value_counts()
colors_sentiment = {'Positive': '#2ecc71', 'Neutral': '#f39c12', 'Negative': '#e74c3c'}
plot_colors = [colors_sentiment[s] for s in sentiment_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(sentiment_counts.index, sentiment_counts.values,
                   color=plot_colors, edgecolor='black', linewidth=0.8)
for bar, count in zip(bars, sentiment_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(count), ha='center', va='bottom', fontsize=13, fontweight='bold')
axes[0].set_title('Sentiment Label Distribution (Bar Chart)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_facecolor('#f8f9fa')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Pie chart
axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index,
            colors=plot_colors, autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'black', 'linewidth': 0.8},
            textprops={'fontsize': 12})
axes[1].set_title('Sentiment Label Distribution (Pie Chart)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('Screenshots/sentiment_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Sentiment distribution plots saved.')

df[['reviewText', 'overall', 'sentiment', 'cleanedText']].head()

## 7. ✂️ Train-Test Split

We split the dataset into **80% training** and **20% testing** sets. A fixed `random_state` ensures reproducibility.

In [ ]:
# Features (cleaned review text) and Labels (sentiment)
X = df['cleanedText']
y = df['sentiment']

# Split: 80% train, 20% test, stratified to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y       # Ensures proportional class distribution in both splits
)

print('✅ Train-Test Split Complete!')
print(f'\nTotal samples  : {len(df)}')
print(f'Training set   : {len(X_train)} samples ({len(X_train)/len(df)*100:.0f}%)')
print(f'Test set       : {len(X_test)} samples ({len(X_test)/len(df)*100:.0f}%)')
print(f'\nClass distribution in training set:')
print(y_train.value_counts())
print(f'\nClass distribution in test set:')
print(y_test.value_counts())

## 8. 🔢 Feature Engineering — TF-IDF Vectorization

### What is TF-IDF?

**TF-IDF** stands for **Term Frequency – Inverse Document Frequency**. It is a numerical statistic used to reflect the importance of a word in a document relative to a collection of documents (corpus).

- **TF (Term Frequency):** How often a word appears in a single document. Words that appear more frequently get higher scores.
- **IDF (Inverse Document Frequency):** Penalizes words that appear in almost every document (e.g., 'the', 'is'), making rare and distinctive words more important.
- **TF-IDF Score = TF × IDF**

**Why TF-IDF?**
- Converts raw text into numerical feature vectors
- Handles the importance of words automatically
- Works very well with linear classifiers like Logistic Regression
- Fast and efficient for beginners

> **Key parameter:** `max_features=5000` — We use only the top 5,000 most important words to keep the feature space manageable while retaining the most informative terms.

In [ ]:
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,     # Use top 5000 most important words
    ngram_range=(1, 2),    # Use unigrams and bigrams (word pairs)
    sublinear_tf=True      # Apply log normalization to term frequency
)

# Fit on training data and transform both train and test sets
# IMPORTANT: Only fit on training data to prevent data leakage
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print('✅ TF-IDF Vectorization complete!')
print(f'\nTraining feature matrix shape : {X_train_tfidf.shape}')
print(f'Test feature matrix shape     : {X_test_tfidf.shape}')
print(f'Vocabulary size               : {len(tfidf_vectorizer.vocabulary_)} unique terms')

# Show top 20 most important features
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f'\n🔑 Sample features (first 20): {list(feature_names[:20])}')

## 9. 🤖 Model Training — Logistic Regression

**Logistic Regression** is a supervised classification algorithm. Despite its name, it is a powerful **classifier** — not a regression model.

Why Logistic Regression for text?
- Works extremely well with high-dimensional, sparse TF-IDF feature vectors
- Highly interpretable — feature coefficients tell us which words push toward each class
- Fast to train, even on large datasets
- Handles multi-class classification natively

In [ ]:
# Initialize the Logistic Regression model
model = LogisticRegression(
    max_iter=1000,      # Maximum iterations for the solver to converge
    random_state=42,    # For reproducibility
    C=1.0,              # Regularization strength (lower = stronger regularization)
    solver='lbfgs',     # Efficient optimizer for multi-class problems
    multi_class='auto'  # Automatically handle multi-class classification
)

# Train the model on the TF-IDF training features
print('⏳ Training Logistic Regression model...')
model.fit(X_train_tfidf, y_train)

print('✅ Model training complete!')

## 10. 📊 Model Evaluation

We evaluate the model using multiple metrics to get a comprehensive view of its performance:

- **Accuracy:** % of correctly predicted reviews
- **Precision:** When the model predicts Positive, how often is it correct?
- **Recall:** Of all actual Positive reviews, how many did the model catch?
- **F1 Score:** Harmonic mean of Precision and Recall — balances both
- **Confusion Matrix:** Visual breakdown of correct and incorrect predictions per class

In [ ]:
# Generate predictions on the test set
y_pred = model.predict(X_test_tfidf)

# ── Compute Evaluation Metrics ────────────────────────────────────────────────
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')

print('=' * 55)
print('          📊 MODEL EVALUATION RESULTS')
print('=' * 55)
print(f'  Accuracy   : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  Precision  : {precision:.4f}  ({precision*100:.2f}%)')
print(f'  Recall     : {recall:.4f}  ({recall*100:.2f}%)')
print(f'  F1 Score   : {f1:.4f}  ({f1*100:.2f}%)')
print('=' * 55)

In [ ]:
# ── Detailed Classification Report ────────────────────────────────────────────
print('\n📋 DETAILED CLASSIFICATION REPORT')
print('─' * 55)
labels_order = ['Negative', 'Neutral', 'Positive']
print(classification_report(y_test, y_pred, target_names=labels_order))

In [ ]:
# ── Confusion Matrix Visualization ────────────────────────────────────────────
labels_order = ['Negative', 'Neutral', 'Positive']
cm = confusion_matrix(y_test, y_pred, labels=labels_order)

fig, ax = plt.subplots(figsize=(8, 6))

# Draw the heatmap manually with matplotlib
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)

# Label axes
ax.set_xticks(range(len(labels_order)))
ax.set_yticks(range(len(labels_order)))
ax.set_xticklabels(labels_order, fontsize=12)
ax.set_yticklabels(labels_order, fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
ax.set_ylabel('Actual Label', fontsize=13, fontweight='bold')
ax.set_title('Confusion Matrix — Sentiment Classifier', fontsize=15, fontweight='bold', pad=15)

# Annotate each cell with the count
thresh = cm.max() / 2
for i in range(len(labels_order)):
    for j in range(len(labels_order)):
        ax.text(j, i, str(cm[i, j]),
                ha='center', va='center', fontsize=16, fontweight='bold',
                color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.savefig('Screenshots/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Confusion matrix saved.')

In [ ]:
# ── Metrics Summary Bar Chart ─────────────────────────────────────────────────
metrics = {
    'Accuracy'  : accuracy,
    'Precision' : precision,
    'Recall'    : recall,
    'F1 Score'  : f1
}

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']
bars = ax.bar(metrics.keys(), metrics.values(), color=bar_colors, edgecolor='black', linewidth=0.8)

# Add percentage labels on bars
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylim(0, 1.15)
ax.set_title('Model Performance Metrics', fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel('Score', fontsize=13)
ax.axhline(y=0.9, color='red', linestyle='--', alpha=0.4, label='90% threshold')
ax.legend(fontsize=10)
ax.set_facecolor('#f8f9fa')
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('Screenshots/metrics_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Metrics summary chart saved.')

## 11. 💬 Interactive Prediction Function

This section allows you to **type your own review** and instantly get the predicted sentiment!

The prediction pipeline:
1. Preprocess the input text (same cleaning steps as training)
2. Transform it using the fitted TF-IDF vectorizer
3. Predict the sentiment class using the trained model
4. Display the result with confidence scores

In [ ]:
def predict_sentiment(review_text, verbose=True):
    """
    Predict the sentiment of a customer review.

    Parameters:
    -----------
    review_text : str
        Raw customer review text.
    verbose : bool
        If True, print a formatted output with confidence scores.

    Returns:
    --------
    str : Predicted sentiment label ('Positive', 'Neutral', or 'Negative')
    """
    # Step 1: Preprocess the input text
    cleaned_review = preprocess_text(review_text)

    # Step 2: Transform using the fitted TF-IDF vectorizer
    review_tfidf = tfidf_vectorizer.transform([cleaned_review])

    # Step 3: Predict class label
    prediction = model.predict(review_tfidf)[0]

    # Step 4: Get probability scores for each class
    probabilities = model.predict_proba(review_tfidf)[0]
    class_labels  = model.classes_

    if verbose:
        emoji_map = {'Positive': '✅ Positive', 'Neutral': '😐 Neutral', 'Negative': '❌ Negative'}
        print('─' * 60)
        print(f'📝 Review   : "{review_text[:80]}..."' if len(review_text) > 80 else f'📝 Review   : "{review_text}"')
        print(f'\n🎯 Predicted Sentiment: {emoji_map[prediction]}')
        print('\n📊 Confidence Scores:')
        for label, prob in sorted(zip(class_labels, probabilities), key=lambda x: -x[1]):
            bar = '█' * int(prob * 20)
            print(f'   {label:<10} : {bar:<20} {prob*100:.1f}%')
        print('─' * 60)

    return prediction


print('✅ Prediction function ready!')

In [ ]:
# ── Example Predictions ───────────────────────────────────────────────────────
print('🧪 EXAMPLE PREDICTIONS')
print('=' * 60)

example_reviews = [
    "This product is absolutely amazing! Works perfectly and looks great.",
    "Terrible quality. Broke after two days. Complete waste of money!",
    "It's okay, nothing special. Average quality, does the job.",
    "Best purchase I made this year! Highly recommend to everyone!",
    "Not what I expected. The description was completely misleading.",
    "Decent product for the price. Nothing really stands out.",
    "Absolutely horrible! Do not buy this garbage. Total scam!",
    "Great value for money! Five stars, arrived quickly and works great.",
]

for review in example_reviews:
    predict_sentiment(review, verbose=True)

In [ ]:
# ── Interactive Prediction ────────────────────────────────────────────────────
# Uncomment and run this cell to enter your own review

print('💬 INTERACTIVE SENTIMENT PREDICTION')
print('=' * 60)

review = input('Enter customer review: ')

if review.strip():
    predict_sentiment(review, verbose=True)
else:
    print('⚠️  No input provided. Please enter a review.')

## 12. 📝 Conclusion

### Summary

In this project, we successfully built an end-to-end **Customer Review Sentiment Analyzer** using classical NLP techniques and machine learning. Here is a recap of what we accomplished:

| Step | Task | Outcome |
|------|------|---------|
| 1 | Data Loading | Loaded Amazon product review dataset with text + ratings |
| 2 | EDA | Explored distributions, found balanced classes |
| 3 | Preprocessing | Cleaned text: removed URLs, HTML, punctuation, stopwords; lemmatized |
| 4 | Label Creation | Mapped star ratings → Positive / Neutral / Negative |
| 5 | Feature Engineering | TF-IDF vectorization (5000 features, unigrams + bigrams) |
| 6 | Model Training | Trained Logistic Regression with `random_state=42` |
| 7 | Evaluation | Achieved strong accuracy, precision, recall, and F1 scores |
| 8 | Prediction | Built interactive function for real-time sentiment prediction |

### Key Takeaways

- **TF-IDF + Logistic Regression** is a powerful, fast, and interpretable baseline for text classification tasks.
- **Text preprocessing** is critical — removing noise dramatically improves model performance.
- **Stratified splitting** ensures class balance is maintained across train and test sets.
- The model generalizes well to unseen reviews, demonstrating its practical utility.

### 🚀 Future Improvements

Here are some directions to extend this project:

1. **Scale Up** — Train on the full Amazon Reviews dataset (millions of reviews) for even better generalization.
2. **Advanced Models** — Experiment with Random Forest, SVM, XGBoost, or ensemble methods.
3. **Deep Learning** — Use LSTM, GRU, or CNN-based text classifiers.
4. **Transformers** — Fine-tune BERT or RoBERTa for state-of-the-art sentiment analysis.
5. **Aspect-Based Sentiment** — Detect sentiment on specific product aspects (price, quality, delivery).
6. **Web App** — Deploy as a Streamlit or Flask application for real-time predictions.
7. **Multilingual Support** — Extend to non-English reviews using multilingual transformers.
8. **Explainability** — Use LIME or SHAP to explain individual predictions.

---

> **Contributed to [ML-CaPsule](https://github.com/Ananya-vastare/ML-CaPsule) under GSSoC'26 | Issue #1921**